# Project 8 — Machine Translation

## Goal
Translate English sentences into another language.

## Three approaches

| Part | Approach | When to use |
|---|---|---|
| **A** | Pre-trained Hugging Face model | Production / real apps |
| **B** | Seq2Seq (Encoder-Decoder LSTM) from scratch | Learn how MT works |
| **C** | Word-by-word dictionary | Toy demo only |

## A note on real MT
Modern translators (Google Translate, DeepL) use **Transformer** architectures with **attention**, trained on millions of bilingual sentence pairs. We won't reproduce that — but we'll see the core idea.

## Part A — Pre-trained translation (Hugging Face)

Install once:
```bash
pip install transformers sentencepiece torch
```

We use Helsinki-NLP's MarianMT models. Swap the language code at the end:
- `Helsinki-NLP/opus-mt-en-fr` → French
- `Helsinki-NLP/opus-mt-en-hi` → Hindi
- `Helsinki-NLP/opus-mt-en-es` → Spanish
- `Helsinki-NLP/opus-mt-en-de` → German

In [ ]:
from transformers import pipeline

en_fr = pipeline('translation', model='Helsinki-NLP/opus-mt-en-fr')

sentences = [
    'Hello, how are you?',
    'Machine translation is fascinating.',
    'I love learning natural language processing.',
]
for s in sentences:
    print(f'EN: {s}')
    print(f'FR: {en_fr(s)[0]["translation_text"]}\n')

## Part B — A Seq2Seq translator from scratch

### Architecture
```
ENGLISH SENTENCE  →  [ENCODER LSTM]  →  thought vector
                                              │
                                              ▼
                       [DECODER LSTM]  →  FRENCH SENTENCE
                                          (one char at a time)
```

The **encoder** reads the input and outputs a hidden state.
The **decoder** uses that state to generate the output one symbol at a time.

We use a tiny toy dataset and **character-level** modelling so the model fits in seconds.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense

pairs = [
    ('hello', 'bonjour'),
    ('good morning', 'bonjour'),
    ('good night', 'bonne nuit'),
    ('how are you', 'comment ca va'),
    ('i am fine', 'je vais bien'),
    ('thank you', 'merci'),
    ('yes', 'oui'),
    ('no', 'non'),
    ('i love you', 'je t aime'),
    ('see you tomorrow', 'a demain'),
    ('good bye', 'au revoir'),
    ('please', 's il vous plait'),
]

### Build character vocabularies
We add `\t` as the START token and `\n` as the END token of the target.

In [ ]:
input_texts = [s for s, _ in pairs]
target_texts = [f'\t{t}\n' for _, t in pairs]

input_chars  = sorted(set(''.join(input_texts)))
target_chars = sorted(set(''.join(target_texts)))

input_char_index  = {c: i for i, c in enumerate(input_chars)}
target_char_index = {c: i for i, c in enumerate(target_chars)}
reverse_target_char_index = {i: c for c, i in target_char_index.items()}

n_enc, n_dec = len(input_chars), len(target_chars)
max_enc = max(len(t) for t in input_texts)
max_dec = max(len(t) for t in target_texts)
print(n_enc, n_dec, max_enc, max_dec)

### One-hot encode
Each character becomes a one-hot vector. Shapes: `(n_pairs, max_len, n_chars)`.

In [ ]:
encoder_input  = np.zeros((len(pairs), max_enc, n_enc), dtype='float32')
decoder_input  = np.zeros((len(pairs), max_dec, n_dec), dtype='float32')
decoder_target = np.zeros((len(pairs), max_dec, n_dec), dtype='float32')

for i, (inp, tgt) in enumerate(zip(input_texts, target_texts)):
    for t, c in enumerate(inp):
        encoder_input[i, t, input_char_index[c]] = 1.0
    for t, c in enumerate(tgt):
        decoder_input[i, t, target_char_index[c]] = 1.0
        if t > 0:
            decoder_target[i, t-1, target_char_index[c]] = 1.0
print('Tensors ready:', encoder_input.shape, decoder_input.shape)

### Build the seq2seq model

In [ ]:
LATENT = 64

encoder_inputs = Input(shape=(None, n_enc))
_, sh, sc = LSTM(LATENT, return_state=True)(encoder_inputs)
encoder_states = [sh, sc]

decoder_inputs = Input(shape=(None, n_dec))
dec_lstm = LSTM(LATENT, return_sequences=True, return_state=True)
dec_out, _, _ = dec_lstm(decoder_inputs, initial_state=encoder_states)
dec_dense = Dense(n_dec, activation='softmax')
dec_out = dec_dense(dec_out)

model = Model([encoder_inputs, decoder_inputs], dec_out)
model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
model.fit([encoder_input, decoder_input], decoder_target,
          batch_size=8, epochs=200, verbose=0)
print('Training done.')

### Inference
At prediction time the decoder runs **one step at a time** because it doesn't see the target. So we split into separate encoder & decoder models.

In [ ]:
encoder_model = Model(encoder_inputs, encoder_states)

dec_state_h = Input(shape=(LATENT,))
dec_state_c = Input(shape=(LATENT,))
dec_states_in = [dec_state_h, dec_state_c]
dec_out2, h2, c2 = dec_lstm(decoder_inputs, initial_state=dec_states_in)
dec_out2 = dec_dense(dec_out2)
decoder_model = Model([decoder_inputs] + dec_states_in, [dec_out2, h2, c2])

def translate(text):
    x = np.zeros((1, max_enc, n_enc), dtype='float32')
    for t, c in enumerate(text):
        if c in input_char_index:
            x[0, t, input_char_index[c]] = 1.0
    states = encoder_model.predict(x, verbose=0)
    target = np.zeros((1, 1, n_dec))
    target[0, 0, target_char_index['\t']] = 1.0
    out = ''
    while True:
        o, h, c = decoder_model.predict([target] + states, verbose=0)
        idx = np.argmax(o[0, -1, :])
        ch = reverse_target_char_index[idx]
        if ch == '\n' or len(out) > max_dec:
            break
        out += ch
        target = np.zeros((1, 1, n_dec)); target[0, 0, idx] = 1.0
        states = [h, c]
    return out

for s in ['hello', 'thank you', 'i love you', 'good night']:
    print(f'EN: {s:<20} → FR: {translate(s)}')

## Part C — Word-by-word dictionary translator (toy)

Just to demonstrate spaCy tokenization. **Not** a real translator.

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')

en_to_fr = {
    'i': 'je', 'you': 'tu', 'he': 'il', 'she': 'elle',
    'love': 'aime', 'hate': 'déteste', 'eat': 'mange',
    'apple': 'pomme', 'book': 'livre', 'cat': 'chat',
    'dog': 'chien', 'the': 'le', 'a': 'un', 'is': 'est',
    'good': 'bon', 'bad': 'mauvais',
}

def naive_translate(sentence):
    doc = nlp(sentence.lower())
    out = []
    for tok in doc:
        if tok.is_punct or tok.is_space: continue
        out.append(en_to_fr.get(tok.text, tok.text))
    return ' '.join(out)

for s in ['I love the cat', 'She is good', 'The dog eats a apple']:
    print(f'EN: {s}')
    print(f'FR: {naive_translate(s)}\n')

## Summary

- For real-world translation: use **pre-trained Transformer models** (Hugging Face)
- For learning the theory: build a **Seq2Seq Encoder-Decoder** with LSTMs
- Modern MT replaces LSTM with **Transformer + attention** — that's the next frontier (BERT, GPT, T5)